# 🧬 Analisi Numerica — Regressione per la Stima del Decadimento Muscolare

Questo notebook esegue l'intera pipeline del progetto direttamente su **Google Colab**,
importando i moduli algoritmici dalla cartella `src/` del repository.

**Prerequisito:** il repository `biological-data-regression` deve essere caricato nel tuo Google Drive.

---

## 0 — Setup: Monta Google Drive e configura i path

In [ ]:
# === SETUP: Montaggio Google Drive e configurazione percorsi ===
# Questa cella va eseguita UNA SOLA VOLTA all'apertura del notebook.
# Ti verra' chiesto di autorizzare l'accesso al tuo Drive.

from google.colab import drive
drive.mount('/content/drive')

import sys
import os

# MODIFICA QUESTO PATH se la tua cartella e' in una posizione diversa nel Drive
PROJECT_DIR = "/content/drive/MyDrive/progetto Lucia/biological-data-regression"

# Aggiunge il progetto al path di Python per poter importare i moduli in src/
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f"Directory di lavoro: {os.getcwd()}")
print(f"Contenuto: {os.listdir('.')}")

In [ ]:
# Installazione delle dipendenze (necessario solo alla prima esecuzione)
!pip install -q numpy pandas matplotlib scipy certifi

In [ ]:
# Configurazione Matplotlib per visualizzare i grafici inline
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50

# Path al dataset e alla cartella dei plot
DATASET_PATH = "data/raw/ncbi_muscular_dystrophy_raw.csv"
PLOTS_DIR = None  # None = mostra i grafici inline senza salvarli su disco
TARGET_COL = "target_expression"

# Verifica che il dataset esista
if os.path.exists(DATASET_PATH):
    print(f"Dataset trovato: {DATASET_PATH}")
else:
    print(f"Dataset non trovato. Esegui la cella di Data Ingestion qui sotto.")

### (Opzionale) Download del dataset da NCBI GEO

Esegui questa cella **solo se** il dataset `ncbi_muscular_dystrophy_raw.csv` non e' presente nella cartella `data/raw/`.
Lo script si colleghera' automaticamente al database NCBI GEO e scarichera' i dati di espressione genica.

In [ ]:
# Esegui SOLO se il dataset non e' gia' presente
if not os.path.exists(DATASET_PATH):
    from src.data_ingestion import run_ingestion
    run_ingestion()
    print("\nDataset scaricato e salvato.")
else:
    print("Dataset gia' presente, skip.")

---
## FASE 1 — Analisi 0-Dimensionale

Calcolo degli indicatori statistici descrittivi (media, mediana, varianza, std),
intervalli al 90%, matrice di covarianza e visualizzazioni:
- Bar plot degli indicatori
- Box plot affiancati
- ECDF vs Gaussiana teorica

In [ ]:
from src.data_exploration import run_phase1

df, stats_df, cov_df = run_phase1(
    dataset_path=DATASET_PATH,
    plots_dir=PLOTS_DIR,
)

---
## FASE 2 — Regressione Lineare Base  y = ax + b

Selezione del predittore migliore tramite matrice di covarianza,
costruzione algebrica della retta di regressione con i minimi quadrati
e scatter plot con retta sovrapposta.

In [ ]:
from src.linear_regression import run_phase2

predictor_col, a, b, results_df = run_phase2(
    df=df,
    target_col=TARGET_COL,
    cov_df=cov_df,
    plots_dir=PLOTS_DIR,
)

print(f"\nModello costruito: y = {a:.4f} * x + ({b:.4f})")
print(f"Predittore selezionato: {predictor_col}")

---
## FASE 3 — Train / Test Split e Validazione

Partizionamento 90% Training / 10% Test con seed fisso (42).
Addestramento del modello sul training set e validazione oggettiva sul test set.

In [ ]:
from src.linear_regression import run_phase3, predict
from src.metrics import mse, rmse, r_squared

a_train, b_train, (x_tr, x_te, y_tr, y_te) = run_phase3(
    df=df,
    predictor_col=predictor_col,
    target_col=TARGET_COL,
    plots_dir=PLOTS_DIR,
)

# Salva le metriche della regressione semplice per il confronto in Fase 5
y_pred_tr = predict(x_tr, a_train, b_train)
y_pred_te = predict(x_te, a_train, b_train)
simple_metrics = {
    "mse_train":  mse(y_tr, y_pred_tr),
    "rmse_train": rmse(y_tr, y_pred_tr),
    "r2_train":   r_squared(y_tr, y_pred_tr),
    "mse_test":   mse(y_te, y_pred_te),
    "rmse_test":  rmse(y_te, y_pred_te),
    "r2_test":    r_squared(y_te, y_pred_te),
}

---
## FASE 4 — Interpolazione ed Estrapolazione

Predizione in punti non osservati: punto medio delle ascisse, inversione della retta,
estrapolazione per x = 0 e input utente.

**Nota Colab:** La fase 4 include un prompt interattivo. Su Colab
apparira' un campo di testo in cui potrai digitare il tuo valore.

In [ ]:
from src.linear_regression import run_phase4

run_phase4(
    df=df,
    predictor_col=predictor_col,
    target_col=TARGET_COL,
    a=a_train,
    b=b_train,
    plots_dir=PLOTS_DIR,
)

---
## FASE 5 — Modello Multilineare  y = X * theta

Regressione multilineare utilizzando **tutte** le feature continue del dataset.
Calcolo di theta tramite le Equazioni Normali: theta = (X^T X)^{-1} X^T y.
Confronto diretto con il modello lineare semplice della Fase 2/3.

In [ ]:
from src.multivariate import run_phase5

theta, res_train, res_test = run_phase5(
    df=df,
    target_col=TARGET_COL,
    plots_dir=PLOTS_DIR,
    lambda_reg=0.0,
    simple_metrics=simple_metrics,
)

---
## FASE 6 — Pulizia Outlier Dinamica

Loop interattivo che individua il dato con residuo massimo e chiede se rimuoverlo.

**Nota Colab:** Il prompt funziona su Colab. Rispondi `s` per rimuovere
l'outlier proposto, `n` per accettare il dataset e uscire dal loop.

In [ ]:
from src.linear_regression import run_phase6

df_clean = run_phase6(
    df=df,
    predictor_col=predictor_col,
    target_col=TARGET_COL,
    plots_dir=PLOTS_DIR,
)

---
## Pipeline Completata

Tutte le 6 fasi sono state eseguite con successo.

**Riepilogo dei moduli utilizzati:**

| Modulo | Fase | Funzione principale |
|---|---|---|
| `src/data_exploration.py` | Fase 1 | `run_phase1()` |
| `src/linear_regression.py` | Fase 2, 3, 4, 6 | `run_phase2()`, `run_phase3()`, `run_phase4()`, `run_phase6()` |
| `src/multivariate.py` | Fase 5 | `run_phase5()` |
| `src/metrics.py` | Trasversale | `mse()`, `rmse()`, `r_squared()`, `residuals()` |
| `src/plot_utils.py` | Trasversale | `base_fig()`, `save_or_show()` |